# Hierarchical nets

A *substitution transition* in a top-level net stands for a whole subnet. `flatten` replaces it with the subnet's places and transitions, fusing the subnet's input/output ports with the places around the substitution. Internal names are prefixed `<substitution>.`.

```mermaid
flowchart LR
    subgraph top
      Ordered((Ordered 1)) --> Pay[[Pay]] --> Paid((Paid)) --> ship[ship] --> Shipped((Shipped))
    end
    subgraph pay
      in((in)) --> authorise[authorise] --> mid((mid)) --> capture[capture] --> out((out))
    end
```

In [1]:
from petrilab.petri import analysis
from petrilab.petri.hierarchical import flatten, order_top, pay_subnet
from petrilab.petri.invariants import is_place_invariant
from petrilab.petri.snakes_bridge import explore, to_snakes

flat = flatten(order_top(), {"Pay": pay_subnet()})
print(sorted(flat.places))
print(sorted(flat.transitions))
print(flat.transitions["Pay.authorise"].pre, flat.transitions["Pay.capture"].post)

['Ordered', 'Paid', 'Pay.mid', 'Shipped']
['Pay.authorise', 'Pay.capture', 'ship']
{'Ordered': 1} {'Paid': 1}


Port `in` is fused with `Ordered` and `out` with `Paid`; only `mid` becomes a new place (`Pay.mid`). The substitution transition `Pay` disappears, replaced by `Pay.authorise` and `Pay.capture`.

In [2]:
g = flat.reachable()
print("states:", len(g))
print("dead:", [flat.marking_dict(m) for m, s in g.items() if not s])
space = explore(to_snakes(flat))
print("safe:", analysis.is_safe(space))
print("one token total:", is_place_invariant(flat, dict.fromkeys(flat.places, 1)))

states: 4
dead: [{'Ordered': 0, 'Paid': 0, 'Shipped': 1, 'Pay.mid': 0}]
safe: True
one token total: True


The flattened net is a 4-state chain ending in `Shipped=1`. It is 1-safe and the total number of tokens is invariant, which matches the intended behaviour of the hierarchical model.